In [0]:
import os

base = '/Workspace/Users/jcesarsantana215@gmail.com/Intensivo-SQL/datasets'

for nome in ['tabela_clientes', 'tabela_produtos', 'tabela_vendas']:
    caminho = os.path.join(base, f'{nome}.csv')
    spark.read.csv(caminho, header=True, inferSchema=True).createOrReplaceTempView(nome)

print('Temp views criadas: tabela_clientes, tabela_produtos, tabela_vendas')

# SELECT FROM

In [0]:
%sql
SELECT * FROM tabela_produtos

In [0]:
%sql
SELECT * FROM tabela_vendas

In [0]:
%sql
SELECT * FROM tabela_clientes LIMIT 10 --mostra somente os 10 primeiros registros

In [0]:
%sql
SELECT DISTINCT categoria FROM tabela_produtos -- mostra somente os valores que nao se repetem

# WHERE

In [0]:
%sql
SELECT * FROM tabela_produtos 
WHERE id_produto = 11 --seleciona a linha onde o id_produto = 11

In [0]:
%sql
SELECT * FROM tabela_produtos 
WHERE categoria = 'Beleza' --seleciona a linha onde a categoria = Beleza

In [0]:
%sql
SELECT * FROM tabela_produtos 
WHERE preco_unitario > 100 --seleciona a linha onde o preco eh maior que 100

# ORDER BY

In [0]:
%sql
SELECT * FROM tabela_produtos 
ORDER BY preco_unitario DESC --ordena pela coluna preco_unitario em ordem decrescente

In [0]:
%sql
SELECT * FROM tabela_produtos 
ORDER BY preco_unitario ASC --ordena pela coluna preco_unitario em ordem crescente

# COUNT

In [0]:
%sql
SELECT COUNT(*) AS qtde_de_linhas FROM tabela_vendas --quantidade de linhas

In [0]:
%sql
SELECT COUNT(*) AS qtde_clientes FROM tabela_clientes
WHERE idade < 30

In [0]:
%sql
SELECT COUNT(*) AS qtde_clientes FROM tabela_clientes
WHERE idade = 30

In [0]:
%sql
SELECT COUNT(*) AS qtde_clientes FROM tabela_clientes
WHERE idade > 30

In [0]:
%sql
SELECT COUNT(*) FROM tabela_clientes
WHERE idade BETWEEN 20 AND 30 --clientes com idade entre 20 e 30 anos

In [0]:
%sql
SELECT COUNT(*) FROM tabela_clientes
WHERE idade >= 20

# MAX, MIN, MODE, MEAN, AVG

In [0]:
%sql
SELECT max(idade) FROM tabela_clientes --maior idade

In [0]:
%sql
SELECT min(idade) FROM tabela_clientes --menor idade

In [0]:
%sql
SELECT avg(idade) FROM tabela_clientes --média da idade

In [0]:
%sql
SELECT mean(idade) FROM tabela_clientes --média da idade

In [0]:
%sql
SELECT mode(idade) FROM tabela_clientes --numero mais frequente

# Like

In [0]:
%sql
SELECT * FROM tabela_clientes
WHERE primeiro_nome LIKE 'A%' --pega os indices onde o primeiro nome começa com a letra 'A'
AND ultimo_nome LIKE '%o' --pega os indices onde o ultimo nome termina com a letra 'o'
LIMIT 10

# IS NULL/IS NOT NULL

In [0]:
%sql
SELECT * FROM tabela_clientes
WHERE segundo_nome IS NULL --pega os indices onde o segundo nome é nulo

In [0]:
%sql
SELECT * FROM tabela_clientes
WHERE segundo_nome IS NOT NULL --pega os indices onde o segundo nome não é nulo
LIMIT 10

# GROUP BY

In [0]:
%sql
SELECT categoria, AVG(preco_unitario) AS media_preco
FROM tabela_produtos 
GROUP BY categoria --média do preço ordenada por categoria

# HAVING

In [0]:
%sql
SELECT categoria, AVG(preco_unitario) AS media_preco
FROM tabela_produtos 
GROUP BY categoria
HAVING media_preco > 1000 --média do preço ordenada por categoria, onde a média é maior que 1000

# JOINs

In [0]:
%sql
SELECT
tabela_vendas.id_venda,
tabela_vendas.data_venda,
(tabela_produtos.preco_unitario * tabela_vendas.quantidade) AS total_price

FROM tabela_vendas
JOIN tabela_produtos ON tabela_vendas.id_produto = tabela_produtos.id_produto

### INNER JOIN

In [0]:
%sql
SELECT
    tabela_clientes.primeiro_nome,
    tabela_clientes.ultimo_nome,
    tabela_produtos.nome_produto,
    tabela_produtos.categoria,
    tabela_vendas.quantidade,
    tabela_vendas.data_venda
FROM tabela_vendas
INNER JOIN tabela_clientes ON tabela_vendas.id_cliente = tabela_clientes.id_cliente --junta vendas com clientes
INNER JOIN tabela_produtos ON tabela_vendas.id_produto = tabela_produtos.id_produto --junta vendas com produtos
LIMIT 10


### LEFT JOIN

In [0]:
%sql
SELECT
    tabela_produtos.nome_produto,
    tabela_produtos.categoria,
    tabela_vendas.id_venda,
    tabela_vendas.quantidade
FROM tabela_produtos
LEFT JOIN tabela_vendas ON tabela_produtos.id_produto = tabela_vendas.id_produto --mantém todos os produtos, mesmo os que não têm vendas
ORDER BY tabela_produtos.nome_produto
LIMIT 10


### RIGHT JOIN

In [0]:
%sql
    
SELECT
    tabela_clientes.primeiro_nome,
    tabela_clientes.ultimo_nome,
    tabela_clientes.cidade,
    tabela_vendas.id_venda,
    tabela_vendas.quantidade
FROM tabela_vendas
RIGHT JOIN tabela_clientes ON tabela_vendas.id_cliente = tabela_clientes.id_cliente --mantém todos os clientes, mesmo os que não têm vendas
ORDER BY tabela_clientes.primeiro_nome
LIMIT 10


   
# Subqueries


In [0]:
%sql
SELECT
    primeiro_nome,
    ultimo_nome,
    idade,
    cidade
FROM tabela_clientes
WHERE idade > (SELECT AVG(idade) FROM tabela_clientes) --clientes com idade acima da média
ORDER BY idade DESC


   
# CTEs


In [0]:
%sql
WITH vendas_por_produto AS ( --CTE: calcula o total vendido e a quantidade de vendas por produto
    SELECT
        id_produto,
        SUM(quantidade) AS total_vendido, --soma a quantidade vendida de cada produto
        COUNT(*) AS qtde_vendas --conta quantas vendas cada produto teve
    FROM tabela_vendas
    GROUP BY id_produto --agrupa por produto
)
SELECT
    tabela_produtos.nome_produto,
    tabela_produtos.categoria,
    vendas_por_produto.total_vendido, --total de itens vendidos do produto
    vendas_por_produto.qtde_vendas --número de vendas do produto
FROM tabela_produtos
JOIN vendas_por_produto ON tabela_produtos.id_produto = vendas_por_produto.id_produto --junta os dados da CTE com a tabela de produtos
ORDER BY vendas_por_produto.total_vendido DESC --ordena do maior para o menor total vendido

   
# Views


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_vendas_completas AS --TEMP VIEW porque as tabelas base são temporárias (CSVs)
SELECT
    tabela_vendas.id_venda,
    tabela_vendas.data_venda,
    tabela_clientes.primeiro_nome,
    tabela_clientes.ultimo_nome,
    tabela_produtos.nome_produto,
    tabela_produtos.categoria,
    tabela_produtos.preco_unitario,
    tabela_vendas.quantidade,
    (tabela_produtos.preco_unitario * tabela_vendas.quantidade) AS total_venda
FROM tabela_vendas
INNER JOIN tabela_clientes ON tabela_vendas.id_cliente = tabela_clientes.id_cliente
INNER JOIN tabela_produtos ON tabela_vendas.id_produto = tabela_produtos.id_produto --view com dados de vendas, clientes e produtos combinados

In [0]:
%sql
    
SELECT * FROM vw_vendas_completas --consulta a view criada acima
LIMIT 10


   
# Criação de Tabelas


In [0]:
%sql
    
CREATE OR REPLACE TEMP VIEW tabela_resumo_vendas AS --TEMP VIEW porque as tabelas base são temporárias (CSVs). Para criar uma tabela permanente, seria necessário primeiro carregar os CSVs como tabelas permanentes.
SELECT
    tabela_produtos.categoria,
    COUNT(tabela_vendas.id_venda) AS qtde_vendas,
    SUM(tabela_vendas.quantidade) AS total_itens_vendidos,
    SUM(tabela_produtos.preco_unitario * tabela_vendas.quantidade) AS receita_total
FROM tabela_vendas
INNER JOIN tabela_produtos ON tabela_vendas.id_produto = tabela_produtos.id_produto
GROUP BY tabela_produtos.categoria --tabela permanente com resumo de vendas por categoria


In [0]:
%sql
    
SELECT * FROM tabela_resumo_vendas --consulta a tabela criada acima
ORDER BY receita_total DESC
